# FSD Demo — Qwen3-VL vLLM daemon on a Colab runtime

Run every cell **in order**, top to bottom. This notebook is meant to be run
through the official **Google Colab VS Code extension** (Select Kernel -> Colab ->
T4 GPU runtime), but it also works on colab.research.google.com.

After the last cell prints `PUBLIC_VLLM_URL`, point your Mac backend at it and
demo (instructions in the final cell). **Keep VS Code / this notebook open** for the
whole demo — the runtime dies when the browser tab or VS Code disconnects.

In [3]:
import sys, subprocess, torch

assert torch.cuda.is_available(), "No GPU! Select a T4 (or better) GPU runtime in the Colab kernel picker."
print("GPU:", torch.cuda.get_device_name(0), "|", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1), "GiB")

subprocess.run([sys.executable, "-m", "pip", "install", "-q",
    "vllm>=0.11.0", "transformers>=4.57", "qwen-vl-utils==0.0.14",
    "accelerate", "peft", "bitsandbytes"], check=False)

# Realign torchvision/torchaudio to the installed CUDA build
cu = "cu" + torch.version.cuda.replace(".", "")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--upgrade",
    "--index-url", f"https://download.pytorch.org/whl/{cu}",
    "torchvision", "torchaudio"], check=False)
print("torch", torch.__version__, "| CUDA", torch.version.cuda)

: 

In [1]:
import os

hf_dir = os.path.expanduser("~/.cache/huggingface")
os.makedirs(hf_dir, exist_ok=True)
token_path = os.path.join(hf_dir, "token")

if os.path.exists(token_path):
    print("TF token already on disk — good.")
else:
    token = None
    try:
        from google.colab import userdata
        token = userdata.get("HF_TOKEN")
    except Exception:
        pass
    if not token:
        from getpass import getpass
        token = getpass("Hugging Face token (gated Qwen3-VL model): ")
    with open(token_path, "w") as f:
        f.write(token.strip())
    print("HF token written.")

TF token already on disk — good.


In [2]:
import pathlib, subprocess

daemon = r"""#!/bin/bash
set -e
export HF_TOKEN="$(cat "$HOME/.cache/huggingface/token" 2>/dev/null || true)"
pkill -f "vllm.entrypoints.openai" 2>/dev/null || true
sleep 1
nohup python3 -m vllm.entrypoints.openai.api_server \
  --model Qwen/Qwen3-VL-4B-Instruct \
  --served-model-name qwen3-vl-4b \
  --port 8000 \
  --dtype half --enforce-eager --max-model-len 4096 \
  --gpu-memory-utilization 0.9 --limit-mm-per-prompt '{"image": 1}' \
  --trust-remote-code \
  >/tmp/vllm.log 2>&1 &
echo $! > /tmp/vllm.pid
echo "vLLM daemon PID: $(cat /tmp/vllm.pid)"
"""
pathlib.Path("/content/start_vllm_daemon.sh").write_text(daemon)
subprocess.run(["bash", "/content/start_vllm_daemon.sh"], check=True)
print("daemon launching...")
print("NOTE: first start downloads Qwen3-VL-4B-Instruct (~2.4GB) + loads it; allow 3-10 min.")

FileNotFoundError: [Errno 2] No such file or directory: '/content/start_vllm_daemon.sh'

In [ ]:
import subprocess, time

ready = False
for i in range(60):
    r = subprocess.run(["curl", "-s", "-o", "/dev/null", "-w", "%{http_code}",
                        "http://localhost:8000/v1/models"], capture_output=True, text=True, timeout=30)
    if r.stdout.strip() == "200":
        ready = True
        print(f"READY after ~{i*15}s — daemon is serving qwen3-vl-4b")
        break
    if i % 4 == 0:
        tail = subprocess.run(["tail", "-2", "/tmp/vllm.log"], capture_output=True, text=True).stdout.strip()
        print(f"[{(i+1)*15}s] http={r.stdout.strip() or 'n/a'} | {tail[-160:]}")
    time.sleep(15)
if not ready:
    print("NOT READY after 15 min — inspect:")
    print(subprocess.run(["bash", "-c", "tail -30 /tmp/vllm.log"], capture_output=True, text=True).stdout)

In [ ]:
# Expose the daemon with a free cloudflared quick tunnel
import subprocess, time, re

subprocess.run(["bash", "-c",
    "curl -sL https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 "
    "-o /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared"], check=True)
print("cloudflared", subprocess.run(["cloudflared", "--version"], capture_output=True, text=True).stdout.strip())

subprocess.run(["bash", "-c",
    "pkill -f 'cloudflared tunnel' 2>/dev/null; "
    "nohup cloudflared tunnel --url http://localhost:8000 --no-autoupdate "
    ">/tmp/cloudflared.log 2>&1 &"], check=False)

url = None
for i in range(30):
    try:
        txt = open("/tmp/cloudflared.log").read()
    except FileNotFoundError:
        txt = ""
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", txt)
    if m:
        url = m.group(0)
        break
    time.sleep(2)

if url:
    print("PUBLIC_VLLM_URL =", url)
    print("On your Mac backend:")
    print(f'  COLAB_VLLM_URL={url}/v1/chat/completions')
else:
    print("Tunnel not up yet; log tail:")
    print(open("/tmp/cloudflared.log").read()[-600:])

## Now, on your Mac

1. Restart the backend with the tunnel URL from the cell above:

   ```bash
   COLAB_VLLM_URL=https://<your-tunnel>.trycloudflare.com/v1/chat/completions \
     /Users/robotjang/fsd-web-demo/backend/venv/bin/python -m uvicorn main:app \
     --host 0.0.0.0 --port 8000 \
     --app-dir /Users/robotjang/fsd-web-demo/backend
   ```

2. Open `http://localhost:5173` and press start.
3. The bridge now sends frames straight to the Colab daemon over HTTPS
   (base64 data URL) — no colab CLI, no session names.

**Lifetime:** the runtime (and tunnel URL) live exactly as long as the
notebook kernel is connected — keep the VS Code window open. To shut down and
stop billing, just disconnect the Colab kernel or close the notebook.